In [2]:
# 🚀 GPU Cleanup Cell — run first in every session

import gc
import torch
import os

def cleanup_gpu():
    """
    Fully clears GPU memory and Python garbage before reloading models.
    Safe for repeated runs.
    """
    print("🧹 Cleaning up GPU and memory...")

    # Delete all global variables except imports
    for name in list(globals().keys()):
        if name not in ["gc", "torch", "os", "cleanup_gpu"]:
            try:
                del globals()[name]
            except Exception:
                pass

    # Garbage collection
    gc.collect()

    # CUDA cleanup (if GPU is available)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print(f"✅ GPU cache cleared — {torch.cuda.get_device_name(0)} now free.")
    else:
        print("⚠️ No CUDA GPU detected, skipped GPU cleanup.")

cleanup_gpu()

🧹 Cleaning up GPU and memory...
✅ GPU cache cleared — NVIDIA GeForce RTX 4070 Laptop GPU now free.


In [3]:
# ---- Core libraries ----
import chess
import chess.pgn
import chess.engine
import torch

# ---- Transformers for commentary generation ----
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---- Offline text-to-speech ----
import pyttsx3

# ---- Utility libraries ----
import subprocess
import tempfile
from IPython.display import Audio

In [16]:
from huggingface_hub import login

HF_TOKEN = ""
if HF_TOKEN:
    try:
        login(token=HF_TOKEN)
        print("HF login successful.")
    except Exception as e:
        print("HF login failed, proceeding without:", e)
else:
    print("No HF token provided. Proceeding without authentication.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login successful.


In [5]:
# --- Load Text Generator (LLM for commentary, ONLINE) ---

commentary_model_id = "google/gemma-3-1b-it"

print("🌐 Downloading model from Hugging Face...")
tokenizer = AutoTokenizer.from_pretrained(commentary_model_id, local_files_only=False)
text_model = AutoModelForCausalLM.from_pretrained(
    commentary_model_id,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    local_files_only=False,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
text_model.to(device)

# --- Initialize offline Text-to-Speech (pyttsx3) ---
tts_engine = pyttsx3.init()
tts_engine.setProperty('rate', 165)   # speaking speed
tts_engine.setProperty('volume', 0.9) # 0.0 to 1.0

print("✅ Gemma model + Offline TTS engine loaded successfully (online mode).")

🌐 Downloading model from Hugging Face...
✅ Gemma model + Offline TTS engine loaded successfully (online mode).


In [6]:
def generate_commentary(fen: str, move_san: str, engine_hint: str = ""):
    """
    Generate human-like commentary for a chess move.
    """
    prompt = f"""You are a professional chess commentator.
Current position (FEN): {fen}
Move played: {move_san}
Engine evaluation hint: {engine_hint}
Provide a short, clear commentary (2–3 sentences)."""

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = text_model.generate(**inputs, max_new_tokens=100, temperature=0.8, top_p=0.9)
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text.split("Provide a short, clear commentary")[-1].strip()

In [7]:
def speak_commentary(commentary_text: str):
    """
    Convert commentary text to speech using offline pyttsx3.
    """
    print(f"🎙️ Speaking: {commentary_text}\n")
    try:
        tts_engine.say(commentary_text)
        tts_engine.runAndWait()
    except Exception as e:
        print("⚠️ TTS playback failed:", e)

In [8]:
import chess
import subprocess
import os

def evaluate_position(board: chess.Board, stockfish_path="C:\\Users\\Admin\\Downloads\\stockfish\\stockfish-windows-x86-64-avx2.exe", time_limit=0.2):
    """
    Evaluates current board using Stockfish and returns score in centipawns.
    Works in Windows/Jupyter safely (no asyncio).
    """
    try:
        process = subprocess.Popen(
            stockfish_path,
            universal_newlines=True,
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
        )
    except FileNotFoundError:
        raise RuntimeError(
            "❌ Stockfish not found.\n"
            "Download from https://stockfishchess.org/download/ "
            "and set full path in 'stockfish_path' argument."
        )

    # Communicate with engine
    process.stdin.write("uci\n")
    process.stdin.write("isready\n")
    process.stdin.write(f"position fen {board.fen()}\n")
    process.stdin.write(f"go movetime {int(time_limit * 1000)}\n")
    process.stdin.flush()

    best_eval = "0"
    while True:
        line = process.stdout.readline()
        if line == "" or "bestmove" in line:
            break
        if "score cp" in line:
            parts = line.strip().split("score cp ")
            if len(parts) > 1:
                best_eval = parts[1].split()[0]

    process.stdin.write("quit\n")
    process.terminate()

    try:
        return int(best_eval)
    except ValueError:
        return 0


In [9]:
# ---- Play and Comment: Original Stable Version ----

import chess
import time

def play_and_comment(moves=None):
    """
    Plays a short chess sequence, evaluates each position with Stockfish,
    generates commentary for each move, and speaks it aloud.
    """
    if moves is None:
        moves = ["e4", "e5", "Nf3", "Nc6", "Bb5", "a6", "Ba4", "Nf6", "O-O", "Be7"]

    print("\n🎯 Starting Game Commentary...\n")
    board = chess.Board()
    results = []

    for move_san in moves:
        # push move to board
        move = board.parse_san(move_san)
        board.push(move)

        # evaluate with Stockfish
        eval_score = evaluate_position(board)
        engine_hint = f"Engine evaluation: {eval_score/100:.2f} (centipawns)."

        # generate commentary text
        commentary = generate_commentary(board.fen(), move_san, engine_hint)

        # print + speak
        print(f"♟️ Move: {move_san}")
        print(f"💬 {commentary}\n")

        speak_commentary(commentary)
        results.append({
            "move": move_san,
            "eval": eval_score,
            "commentary": commentary
        })
        time.sleep(1)

    print("\n✅ Game commentary completed!")
    return results


# ▶️ Run this to test
moves = ["e4", "e5", "Nf3", "Nc6", "Bb5", "a6", "Ba4", "Nf6", "O-O", "Be7"]
commentary_results = play_and_comment(moves)


🎯 Starting Game Commentary...

♟️ Move: e4
💬 (

🎙️ Speaking: (

♟️ Move: e5
💬 (2–3 sentences).

The position is a very strong, open position. White's move is a classic and principled move, e5, which immediately challenges Black's control of the center. This opens lines for development and attacks.

**Answer:** e5
**Rating:** 0.36
**Comments:** Excellent! e5 is a cornerstone of the opening. It immediately challenges Black's center and prepares for a dynamic and aggressive game.

🎙️ Speaking: (2–3 sentences).

The position is a very strong, open position. White's move is a classic and principled move, e5, which immediately challenges Black's control of the center. This opens lines for development and attacks.

**Answer:** e5
**Rating:** 0.36
**Comments:** Excellent! e5 is a cornerstone of the opening. It immediately challenges Black's center and prepares for a dynamic and aggressive game.

♟️ Move: Nf3
💬 (2–3 sentences).

Okay, let's analyze this opening.

The position is a classic star

In [10]:
import time
from textblob import TextBlob
import numpy as np

def evaluate_performance(results):
    """Compute fluency, polarity, length stats, and timing."""
    fluency_scores, sentiments, lengths = [], [], []

    for r in results:
        text = r["commentary"]
        blob = TextBlob(text)
        fluency_scores.append(len(blob.words))
        sentiments.append(blob.sentiment.polarity)
        lengths.append(len(text.split()))

    summary = {
        "avg_fluency": np.mean(fluency_scores),
        "avg_sentiment": np.mean(sentiments),
        "avg_length": np.mean(lengths),
        "move_count": len(results)
    }

    print("\n📊 Commentary Performance Evaluation")
    print(f"📝 Average sentence length (words): {summary['avg_fluency']:.1f}")
    print(f"💡 Average sentiment polarity: {summary['avg_sentiment']:.2f}")
    print(f"📏 Average total length (words): {summary['avg_length']:.1f}")
    print(f"♟️ Moves processed: {summary['move_count']}")
    return summary


# ▶️ Run after generating commentary_results
evaluation_summary = evaluate_performance(commentary_results)



📊 Commentary Performance Evaluation
📝 Average sentence length (words): 54.0
💡 Average sentiment polarity: 0.09
📏 Average total length (words): 52.0
♟️ Moves processed: 10


In [11]:
# 📊 --- Commentary Performance Evaluation (Fixed Display) ---
from textblob import TextBlob
import numpy as np
import pandas as pd
from IPython.display import display

def evaluate_commentary_performance(results):
    """
    Evaluate linguistic and qualitative metrics for generated commentaries.
    Returns a pandas DataFrame + printed summary.
    """
    if not results or "commentary" not in results[0]:
        print("⚠️ No commentary results found to evaluate.")
        return pd.DataFrame()

    fluency_scores, sentiment_scores, lengths = [], [], []

    for r in results:
        text = r["commentary"]
        blob = TextBlob(text)
        fluency_scores.append(len(blob.words))          # rough fluency = number of words
        sentiment_scores.append(blob.sentiment.polarity)  # -1 to +1
        lengths.append(len(text.split()))

    df = pd.DataFrame({
        "Move": [r["move"] for r in results],
        "Eval (cp)": [r["eval"] for r in results],
        "Length (words)": lengths,
        "Sentiment": sentiment_scores,
        "Fluency (tokens)": fluency_scores,
        "Commentary": [r["commentary"] for r in results],
    })

    summary = {
        "Average length (words)": np.mean(lengths),
        "Average fluency score": np.mean(fluency_scores),
        "Average sentiment polarity": np.mean(sentiment_scores),
        "Moves evaluated": len(results),
    }

    print("\n📊 Commentary Performance Evaluation Summary:")
    for k, v in summary.items():
        print(f"• {k}: {v:.3f}" if isinstance(v, float) else f"• {k}: {v}")

    return df

# ▶️ Run after commentary generation
metrics_df = evaluate_commentary_performance(commentary_results)

# ✅ Explicitly display to avoid KeyError
display(metrics_df)


📊 Commentary Performance Evaluation Summary:
• Average length (words): 52.000
• Average fluency score: 54.000
• Average sentiment polarity: 0.087
• Moves evaluated: 10


,Move,Eval (cp),Length (words),Sentiment,Fluency (tokens),Commentary
0,e4,-35,1,0.000000,0,(
1,e5,36,60,0.072424,63,(2–3 sentences).\n\nThe position is a very str...
2,Nf3,-35,74,0.060000,78,"(2–3 sentences).\n\nOkay, let's analyze this o..."
3,Nc6,36,61,-0.061458,61,"(2–3 sentences).\n\nThe engine has played Nc6,..."
4,Bb5,-44,80,0.042857,85,(2–3 sentences).\n\nThe engine is pushing the ...
5,a6,34,42,0.910000,40,(2–3 sentences).\n\nThe engine is looking for ...
6,Ba4,-35,49,-0.100000,55,(2–3 sentences).\n\nThe engine is pushing a pa...
7,Nf6,29,73,0.210333,75,(2–3 sentences).\n\nThe move is very promising...
8,O-O,-40,38,-0.055556,39,(2–3 sentences).\n\nYour answer should focus o...
9,Be7,32,42,-0.205556,44,"(2–3 sentences).\n\nThe engine plays Be7, whic..."


In [19]:
# 📈 --- NLP Evaluation: BLEU and ROUGE (offline-safe) ---
import numpy as np
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from IPython.display import display

def evaluate_nlp_metrics(results, reference_field="reference", commentary_field="commentary"):
    """
    Evaluate generated commentaries using BLEU and ROUGE metrics.
    Offline-safe version (no Hugging Face or BERTScore dependencies).
    """
    if not results or commentary_field not in results[0]:
        print("⚠️ No commentary results available for NLP evaluation.")
        return pd.DataFrame()

    smooth = SmoothingFunction().method1
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

    bleu_scores, rouge1_scores, rougeL_scores = [], [], []
    references, candidates = [], []

    synthetic_refs = {
        "e4": "White opens with e4, controlling the center and preparing rapid development.",
        "e5": "Black responds symmetrically with e5, contesting the center immediately.",
        "Nf3": "White develops the knight toward the center, attacking e5 and aiding control.",
        "Nc6": "Black develops the knight to defend e5 and prepare for flexible setups.",
        "Bb5": "White enters the Ruy Lopez, increasing tension and pinning the c6 knight.",
        "a6": "Black challenges the bishop, asking it to decide its intentions early.",
        "Ba4": "White retreats to maintain the pin and prepare castling.",
        "Nf6": "Black develops another knight, attacking e4 and completing kingside setup.",
        "O-O": "White castles, securing the king and connecting the rooks.",
        "Be7": "Black finishes development and prepares to castle safely.",
    }

    for r in results:
        move = r["move"]
        cand = r[commentary_field]
        ref = synthetic_refs.get(move, f"A standard move {move} focusing on development and center control.")

        bleu = sentence_bleu([ref.split()], cand.split(), smoothing_function=smooth)
        rouge_scores = rouge.score(ref, cand)
        rouge1 = rouge_scores["rouge1"].fmeasure
        rougeL = rouge_scores["rougeL"].fmeasure

        bleu_scores.append(bleu)
        rouge1_scores.append(rouge1)
        rougeL_scores.append(rougeL)
        references.append(ref)
        candidates.append(cand)

    df = pd.DataFrame({
        "Move": [r["move"] for r in results],
        "BLEU": bleu_scores,
        "ROUGE-1": rouge1_scores,
        "ROUGE-L": rougeL_scores,
        "Commentary": candidates,
        "Reference": references,
    })

    print("\n📊 NLP Metrics Summary:")
    print(f"• Average BLEU: {np.mean(bleu_scores):.3f}")
    print(f"• Average ROUGE-1: {np.mean(rouge1_scores):.3f}")
    print(f"• Average ROUGE-L: {np.mean(rougeL_scores):.3f}")

    return df


# ▶️ Run this after generating `commentary_results`
nlp_metrics_df = evaluate_nlp_metrics(commentary_results)
display(nlp_metrics_df)


📊 NLP Metrics Summary:
• Average BLEU: 0.003
• Average ROUGE-1: 0.085
• Average ROUGE-L: 0.073


,Move,BLEU,ROUGE-1,ROUGE-L,Commentary,Reference
0,e4,0.000000,0.000000,0.000000,(,"White opens with e4, controlling the center an..."
1,e5,0.004001,0.135135,0.108108,(2–3 sentences).\n\nThe position is a very str...,"Black responds symmetrically with e5, contesti..."
2,Nf3,0.003469,0.197802,0.109890,"(2–3 sentences).\n\nOkay, let's analyze this o...","White develops the knight toward the center, a..."
3,Nc6,0.003934,0.111111,0.111111,"(2–3 sentences).\n\nThe engine has played Nc6,...",Black develops the knight to defend e5 and pre...
4,Bb5,0.002982,0.083333,0.083333,(2–3 sentences).\n\nThe engine is pushing the ...,"White enters the Ruy Lopez, increasing tension..."
5,a6,0.004392,0.033898,0.033898,(2–3 sentences).\n\nThe engine is looking for ...,"Black challenges the bishop, asking it to deci..."
6,Ba4,0.004453,0.060606,0.060606,(2–3 sentences).\n\nThe engine is pushing a pa...,White retreats to maintain the pin and prepare...
7,Nf6,0.002958,0.068966,0.068966,(2–3 sentences).\n\nThe move is very promising...,"Black develops another knight, attacking e4 an..."
8,O-O,0.006415,0.122449,0.122449,(2–3 sentences).\n\nYour answer should focus o...,"White castles, securing the king and connectin..."
9,Be7,0.000000,0.034483,0.034483,"(2–3 sentences).\n\nThe engine plays Be7, whic...",Black finishes development and prepares to cas...
